In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]   

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.solver = "cplex"
cobra_config.bounds = -999999.0,999999.0

In [4]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Btheta.tcds.top5.gramNegN.cim8.xml"

In [5]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [6]:
#Fixing masses
model.metabolites.get_by_id("23dhb_c").formula = "C7H6O4"
model.metabolites.get_by_id("2mpdhl_c").formula = "C12H23NO2S2"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("d23hb_e").formula = "C7H7O4"
model.metabolites.get_by_id("db4p_c").formula = "C4H7O6P"
model.metabolites.get_by_id("dmlz_c").formula = "C13H18N4O6"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("fmnRD_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("lipoate_c").formula = "C8H14O2S2"
model.metabolites.get_by_id("lipoate_e").formula = "C8H14O2S2"
model.metabolites.get_by_id("Nforglu_c").formula = "C6H7NO5"
model.metabolites.get_by_id("php2coa_c").formula = "C34H46N7O17P3S"
model.metabolites.get_by_id("phxa2coa_c").formula = "C33H44N7O17P3S"
model.metabolites.get_by_id("pocta2coa_c").formula = "C35H48N7O17P3S"
model.metabolites.get_by_id("ppt2coa_c").formula = "C32H42N7O17P3S"
model.metabolites.get_by_id("R_3hphpcoa_c").formula = "C34H48N7O18P3S"
model.metabolites.get_by_id("R_3hphxacoa_c").formula = "C33H46N7O18P3S"
model.metabolites.get_by_id("R_3hpoctacoa_c").formula = "C35H50N7O18P3S"
model.metabolites.get_by_id("R_3hpptcoa_c").formula = "C32H44N7O18P3S"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"
model.metabolites.get_by_id("salchs4_c").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_e").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4_p").formula = "C42H42N3O25"
model.metabolites.get_by_id("salchs4fe_c").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_e").formula = "C42FeH42N3O25"
model.metabolites.get_by_id("salchs4fe_p").formula = "C42FeH42N3O25"

In [7]:
#Fixing charges
model.metabolites.get_by_id("23dhb_c").charge = 0
model.metabolites.get_by_id("23ddhb_c").charge = -1
model.metabolites.get_by_id("2agpg120_c").charge = -1
model.metabolites.get_by_id("2agpg120_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2maacoa_c").charge = -4
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("3hmrsACP_c").charge = -1
model.metabolites.get_by_id("3padsel_c").charge = -4
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("actACP_c").charge = -1
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("adsel_c").charge = -2
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("alpro_c").charge = 1
model.metabolites.get_by_id("amob_c").charge = 0
model.metabolites.get_by_id("anhgm3p_c").charge = -2
model.metabolites.get_by_id("anhgm3p_p").charge = -2
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("but2eACP_c").charge = -1
model.metabolites.get_by_id("db4p_c").charge = -2
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("dhgly_c").charge = -1
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("dtbt_c").charge = -1
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fc1p_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxox_c").charge = 1
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fe3dcit_c").charge = -3
model.metabolites.get_by_id("fe3dcit_e").charge = -3
model.metabolites.get_by_id("fe3dcit_p").charge = -3
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("focytc_c").charge = 1
model.metabolites.get_by_id("fpram_c").charge = -2
model.metabolites.get_by_id("fruur_c").charge = -1
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("g3pg_c").charge = -1
model.metabolites.get_by_id("g3pg_e").charge = -1
model.metabolites.get_by_id("g3pg_p").charge = -1
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("lipoate_c").charge = 0
model.metabolites.get_by_id("lipoate_e").charge = 0
model.metabolites.get_by_id("lnlccoa_c").charge = -4
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("murein3px3p_p").charge = -4
model.metabolites.get_by_id("murein4px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4px4p_p").charge = -6
model.metabolites.get_by_id("myrsACP_c").charge = -1
model.metabolites.get_by_id("Nforglu_c").charge = -2
model.metabolites.get_by_id("oc2coa_c").charge = -4
model.metabolites.get_by_id("octeACP_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa141_c").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa120_p").charge = -2
model.metabolites.get_by_id("pa140_p").charge = -2
model.metabolites.get_by_id("pa141_p").charge = -2
model.metabolites.get_by_id("pa161_p").charge = -2
model.metabolites.get_by_id("pa180_p").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg140_c").charge = -1
model.metabolites.get_by_id("pg141_c").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg140_p").charge = -1
model.metabolites.get_by_id("pg141_p").charge = -1
model.metabolites.get_by_id("pg160_p").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pgp120_c").charge = -3
model.metabolites.get_by_id("pgp140_c").charge = -3
model.metabolites.get_by_id("pgp141_c").charge = -3
model.metabolites.get_by_id("pgp160_c").charge = -3
model.metabolites.get_by_id("pgp161_c").charge = -3
model.metabolites.get_by_id("pgp180_c").charge = -3
model.metabolites.get_by_id("pgp181_c").charge = -3
model.metabolites.get_by_id("pgp120_p").charge = -3
model.metabolites.get_by_id("pgp140_p").charge = -3
model.metabolites.get_by_id("pgp141_p").charge = -3
model.metabolites.get_by_id("pgp160_p").charge = -3
model.metabolites.get_by_id("pgp161_p").charge = -3
model.metabolites.get_by_id("pgp180_p").charge = -3
model.metabolites.get_by_id("pgp181_p").charge = -3
model.metabolites.get_by_id("ppgpp_c").charge = -6
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("ps161_c").charge = -1
model.metabolites.get_by_id("ps181_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("rml1p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("tagur_c").charge = -1
model.metabolites.get_by_id("td2coa_c").charge = -4
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("tmrs2eACP_c").charge = -1
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2
model.metabolites.get_by_id("udpacgal_c").charge = -2
model.metabolites.get_by_id("vacc_c").charge = -1
model.metabolites.get_by_id("vacccoa_c").charge = -4

In [8]:
# Identifying and removing duplicated reactions
[md,rd, dt] = removeDuplicateRxn(model)

In [9]:
#Removing the duplicated reactions that were in doubt
md.remove_reactions([md.reactions.get_by_id("TMDK1_1"),md.reactions.get_by_id("URIK3_1"),
                     md.reactions.get_by_id("SHKK_1"),md.reactions.get_by_id("IZPN_1"),
                     md.reactions.get_by_id("IG3PS_1"),md.reactions.get_by_id("CYTDK2_1"),
                     md.reactions.get_by_id("3SALATAi"),md.reactions.get_by_id("URIK1_1"),
                     md.reactions.get_by_id("URIK2_1"),md.reactions.get_by_id("CYTDK1_1"),
                     md.reactions.get_by_id("HMPK1_1"),md.reactions.get_by_id("RBK2"),
                     md.reactions.get_by_id("RBFSa_1"),md.reactions.get_by_id("DURIK1_1"),
                     md.reactions.get_by_id("ASPA2")])

md.remove_metabolites([md.metabolites.get_by_id("3sala_c"),md.metabolites.get_by_id("3snpyr_c")])
                     
md.repair()

In [10]:
#Running FVA again after removing reactions
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arg__L_e,EX_arg__L_e,0.07451,[0; 0.265],6,0.25%
asn__L_e,EX_asn__L_e,0.06072,[0; 1.022],4,0.13%
ca2_e,EX_ca2_e,0.001311,[0.001246; 0.001311],0,0.00%
cl_e,EX_cl_e,0.001311,[0.001246; 0.001311],0,0.00%
cobalt2_e,EX_cobalt2_e,2.519E-05,[2.393E-05; 2.519E-05],0,0.00%
cu2_e,EX_cu2_e,0.0001786,[0.0001697; 0.0001786],0,0.00%
cys__L_e,EX_cys__L_e,0.1003,[0; 1.045],3,0.17%
fe2_e,EX_fe2_e,0.003771,[0.001714; 0.01393],0,0.00%
fol_e,EX_fol_e,0.0001685,[0; 0.0001685],19,0.00%
fru_e,EX_fru_e,10,[5.25; 10],6,33.04%


In [11]:
#Loading BiGG's universal model for gapfilling
universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [12]:
#Performing gapfill with universal model from Cobrapy to reduce blocked reactions
gapfill(md, universal, demand_reactions=False, exchange_reactions=True, iterations=10, lower_bound = 1e-9)

[[], [], [], [], [], [], [], [], [], []]

In [13]:
#Adding/Removing/Editing reactions to reduce blocked reactions

md.add_metabolites([universal.metabolites.get_by_id("abt_e")])
md.add_reactions([universal.reactions.get_by_id("ABTt"),universal.reactions.get_by_id("EX_abt_e")]) #Missing transfer to cytoplasm
md.metabolites.get_by_id("abt_e").formula = md.metabolites.get_by_id("abt_c").formula
md.metabolites.get_by_id("abt_e").charge = md.metabolites.get_by_id("abt_c").charge

md.remove_reactions([md.reactions.get_by_id("CAt6pp"),md.reactions.get_by_id("UREAt"),
                     md.reactions.get_by_id("EX_urea_e"),
                    md.reactions.get_by_id("EX_m_xyl_e"),md.reactions.get_by_id("MXYLt5"),
                    md.reactions.get_by_id("M_Xylt1"),md.reactions.get_by_id("MXYLt6"),
                    md.reactions.get_by_id("M_XYLtpp"),md.reactions.get_by_id("EX_met__D_e"),
                    md.reactions.get_by_id("METDtex"),md.reactions.get_by_id("METte"),
                    md.reactions.get_by_id("METDabcpp"),md.reactions.get_by_id("EX_no3_e"),
                    md.reactions.get_by_id("NO3t3"),md.reactions.get_by_id("NO3t7pp"),
                    md.reactions.get_by_id("NO3tex"),md.reactions.get_by_id("EX_p_xyl_e"),
                    md.reactions.get_by_id("P_Xylt1"),md.reactions.get_by_id("PXYLt5"),
                    md.reactions.get_by_id("PXYLt6"),md.reactions.get_by_id("P_XYLtpp"),
                    md.reactions.get_by_id("EX_tol_e"),md.reactions.get_by_id("TOLt6"),
                    md.reactions.get_by_id("TOLt5"),md.reactions.get_by_id("TOLtex"),
                    md.reactions.get_by_id("TOLtpp")]) #Nothing is done with them 
md.remove_metabolites([md.metabolites.get_by_id("ca2_p"),md.metabolites.get_by_id("urea_e"),
                       md.metabolites.get_by_id("urea_c"),md.metabolites.get_by_id("m_xyl_c"),
                       md.metabolites.get_by_id("m_xyl_p"),md.metabolites.get_by_id("m_xyl_e"),
                       md.metabolites.get_by_id("met__D_c"),md.metabolites.get_by_id("met__D_p"),
                       md.metabolites.get_by_id("met__D_e"),md.metabolites.get_by_id("no3_c"),
                       md.metabolites.get_by_id("no3_p"),md.metabolites.get_by_id("no3_e"),
                       md.metabolites.get_by_id("p_xyl_e"),md.metabolites.get_by_id("p_xyl_p"),
                       md.metabolites.get_by_id("p_xyl_c"),md.metabolites.get_by_id("tol_c"),
                      md.metabolites.get_by_id("tol_e"),md.metabolites.get_by_id("tol_p")])

md.remove_reactions([md.reactions.get_by_id("RECOAH10"),md.reactions.get_by_id("RECOAH11"),
                    md.reactions.get_by_id("RECOAH12"),md.reactions.get_by_id("RECOAH13"),
                    md.reactions.get_by_id("RECOAH14"),md.reactions.get_by_id("RECOAH15"),
                    md.reactions.get_by_id("RECOAH16"),md.reactions.get_by_id("RECOAH17"),
                    md.reactions.get_by_id("RECOAH19"),md.reactions.get_by_id("RECOAH20"),
                    md.reactions.get_by_id("RECOAH21"),md.reactions.get_by_id("RECOAH22"),
                    md.reactions.get_by_id("RECOAH23"),md.reactions.get_by_id("RECOAH24"),
                    md.reactions.get_by_id("RECOAH25"),md.reactions.get_by_id("RECOAH26"),
                    md.reactions.get_by_id("RECOAH4"),md.reactions.get_by_id("RECOAH7"),
                    md.reactions.get_by_id("RECOAH9")]) #Not connected to the network
md.remove_metabolites([md.metabolites.get_by_id("R_3hnonacoa_c"),md.metabolites.get_by_id("nona2coa_c"),
                       md.metabolites.get_by_id("2mpdhl_c"),md.metabolites.get_by_id("3gmp_c"),
                       md.metabolites.get_by_id("4atb2coa_c"),md.metabolites.get_by_id("6ath2coa_c"),
                       md.metabolites.get_by_id("R3hdec4coa_c"),md.metabolites.get_by_id("R_3h4atbcoa_c"),
                       md.metabolites.get_by_id("R_3h6athcoa_c"),md.metabolites.get_by_id("R_3hcddec5ecoa_c"),
                       md.metabolites.get_by_id("R_3hcmrs7ecoa_c"),md.metabolites.get_by_id("R_3hdcoa_c"),
                       md.metabolites.get_by_id("R_3hdd6coa_c"),md.metabolites.get_by_id("R_3hhdcoa_c"),
                       md.metabolites.get_by_id("R_3hhpcoa_c"),md.metabolites.get_by_id("R_3hpbcoa_c"),
                       md.metabolites.get_by_id("R_3hpdecacoa_c"),md.metabolites.get_by_id("R_3hphpcoa_c"),
                       md.metabolites.get_by_id("R_3hphxacoa_c"),md.metabolites.get_by_id("R_3hpnonacoa_c"),
                       md.metabolites.get_by_id("R_3hpoctacoa_c"),md.metabolites.get_by_id("R_3hpptcoa_c"),
                       md.metabolites.get_by_id("R_3htd58coa_c"),md.metabolites.get_by_id("R_3htd5coa_c"),
                       md.metabolites.get_by_id("acmana_c"),md.metabolites.get_by_id("cellb_c"),
                       md.metabolites.get_by_id("dd6_2_coa_c"),md.metabolites.get_by_id("dde2coa_c"),
                       md.metabolites.get_by_id("dec4_2_coa_c"),md.metabolites.get_by_id("hp2coa_c"),
                       md.metabolites.get_by_id("indpyr_c"),md.metabolites.get_by_id("nal2a6o_c"),
                       md.metabolites.get_by_id("pb2coa_c"),md.metabolites.get_by_id("pdca2coa_c"),
                       md.metabolites.get_by_id("php2coa_c"),md.metabolites.get_by_id("phxa2coa_c"),
                       md.metabolites.get_by_id("pnona2coa_c"),md.metabolites.get_by_id("pocta2coa_c"),
                       md.metabolites.get_by_id("ppt2coa_c"),md.metabolites.get_by_id("td58_2_coa_c"),
                       md.metabolites.get_by_id("tde2coa_c"),md.metabolites.get_by_id("tded5_2_coa_c")])


In [14]:
#Running FVA again after adding and removing reactions
md.summary(fva=0.95)

Metabolite,Reaction,Flux,Range,C-Number,C-Flux
arg__L_e,EX_arg__L_e,2.192,[0; 9.617],6,6.41%
asn__L_e,EX_asn__L_e,9.738,[0; 10],4,18.99%
ca2_e,EX_ca2_e,0.03857,[0.03664; 0.03857],0,0.00%
cl_e,EX_cl_e,0.03857,[0.03664; 0.03857],0,0.00%
cobalt2_e,EX_cobalt2_e,0.000741,[0.000704; 0.0007404],0,0.00%
cu2_e,EX_cu2_e,0.005254,[0.004991; 0.005254],0,0.00%
cys__L_e,EX_cys__L_e,2.947,[0; 10],3,4.31%
fe2_e,EX_fe2_e,0.1109,[0.05041; 0.4156],0,0.00%
fol_e,EX_fol_e,0.004958,[0; 0.004958],19,0.05%
gln__L_e,EX_gln__L_e,10,[0; 10],5,24.38%


In [15]:
#Loopless FBA with new set of reactions

# Trying to solve Stoichiometrically Balanced Cycles 
rlist = [md.reactions.get_by_id("3HAACOAT140"),md.reactions.get_by_id("3HACPH"),md.reactions.get_by_id("3HAD140"),
         md.reactions.get_by_id("3HAD40"),md.reactions.get_by_id("3HOXTPP"),md.reactions.get_by_id("3OAR40"),
         md.reactions.get_by_id("4HGLSD"),md.reactions.get_by_id("5DGLCNR"),md.reactions.get_by_id("5DKGR"),
         md.reactions.get_by_id("ACOAD1"),md.reactions.get_by_id("ACOAD1f"),md.reactions.get_by_id("ACOAD1fr"),
         md.reactions.get_by_id("ACOAD2"),md.reactions.get_by_id("ACOAD2f"),md.reactions.get_by_id("ACOAD3"),
         md.reactions.get_by_id("ACOAD3f"),md.reactions.get_by_id("ACOAD6"),md.reactions.get_by_id("ACOAD6f"),
         md.reactions.get_by_id("ACOAD7"),md.reactions.get_by_id("ACOAD7f"),md.reactions.get_by_id("ACTD"),
         md.reactions.get_by_id("ACTD_1"),md.reactions.get_by_id("ACTDa"),md.reactions.get_by_id("ADK1"),
         md.reactions.get_by_id("ADK2"),md.reactions.get_by_id("ADK3"),md.reactions.get_by_id("ADK4"),
         md.reactions.get_by_id("ADKd"),md.reactions.get_by_id("AKGDH"),md.reactions.get_by_id("ALAD_L"),
         md.reactions.get_by_id("ALATA_L"),md.reactions.get_by_id("ASPT"),md.reactions.get_by_id("ASPTA"),
         md.reactions.get_by_id("BUTCT2"),md.reactions.get_by_id("BUTKr"),md.reactions.get_by_id("CDDTPP"),
         md.reactions.get_by_id("CO2t"),md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),
         md.reactions.get_by_id("CYTDK1"),md.reactions.get_by_id("CYTDK2"),md.reactions.get_by_id("CYTK11_1"),
         md.reactions.get_by_id("DACTPP"),md.reactions.get_by_id("DADK"),md.reactions.get_by_id("DATCY"),
         md.reactions.get_by_id("DCPP"),md.reactions.get_by_id("DCTCP"),md.reactions.get_by_id("DGTCY"),
         md.reactions.get_by_id("DTTGY"),md.reactions.get_by_id("DUCYTP"),md.reactions.get_by_id("DURIPP"),
         md.reactions.get_by_id("DURIPP_1"),md.reactions.get_by_id("DUTCP"),md.reactions.get_by_id("EAR140x"),
         md.reactions.get_by_id("FFSD1r"),md.reactions.get_by_id("FGFT_1"),md.reactions.get_by_id("FUM"),
         md.reactions.get_by_id("G3PD1ir"),md.reactions.get_by_id("G3PD2"),md.reactions.get_by_id("G6PBDH"),
         md.reactions.get_by_id("G6PDH2r"),md.reactions.get_by_id("G6PI"),md.reactions.get_by_id("GALM1"),
         md.reactions.get_by_id("GARFT"),md.reactions.get_by_id("GCCa"),md.reactions.get_by_id("GCCa_1"),
         md.reactions.get_by_id("GCCb"),md.reactions.get_by_id("GCCb_1"),md.reactions.get_by_id("GCCc"),
         md.reactions.get_by_id("GCCc_1"),md.reactions.get_by_id("GDH1"),md.reactions.get_by_id("GLBRAN2"),
         md.reactions.get_by_id("GLDBRAN2"),md.reactions.get_by_id("GLUDy"),md.reactions.get_by_id("GLYCL"),
         md.reactions.get_by_id("GLYCL_2"),md.reactions.get_by_id("GalMr"),md.reactions.get_by_id("GalMr_2"),
         md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),md.reactions.get_by_id("H2Otpp"),
         md.reactions.get_by_id("HPROa"),md.reactions.get_by_id("HPROb"),md.reactions.get_by_id("HPROx"),
         md.reactions.get_by_id("ICDHyr"),md.reactions.get_by_id("ICITRED"),md.reactions.get_by_id("ITCY"),
         md.reactions.get_by_id("KGD2"),md.reactions.get_by_id("MDH"),md.reactions.get_by_id("MTHFC"),
         md.reactions.get_by_id("NDPK1"),md.reactions.get_by_id("NDPK2"),md.reactions.get_by_id("NDPK4"),
         md.reactions.get_by_id("NDPK5"),md.reactions.get_by_id("NDPK6"),md.reactions.get_by_id("NDPK7"),
         md.reactions.get_by_id("NDPK8"),md.reactions.get_by_id("NDPK9"),md.reactions.get_by_id("NH3c"),
         md.reactions.get_by_id("NH4t"),md.reactions.get_by_id("NH4tex"),md.reactions.get_by_id("NH4tpp"),
         md.reactions.get_by_id("NP1"),md.reactions.get_by_id("NP1_1"),md.reactions.get_by_id("OCOAT1"),
         md.reactions.get_by_id("OSUCCL"),md.reactions.get_by_id("P5CR"),md.reactions.get_by_id("P5CRx"),
         md.reactions.get_by_id("PBUTT"),md.reactions.get_by_id("PHCD"),md.reactions.get_by_id("PHCHGS"),
         md.reactions.get_by_id("PNP"),md.reactions.get_by_id("PNP_1"),md.reactions.get_by_id("PPK2"),
         md.reactions.get_by_id("PROD2"),md.reactions.get_by_id("PSUDS"),md.reactions.get_by_id("PUNP1"),
         md.reactions.get_by_id("PUNP1_1"),md.reactions.get_by_id("PUNP2"),md.reactions.get_by_id("PUNP2_1"),
         md.reactions.get_by_id("PUNP3"),md.reactions.get_by_id("PUNP3_1"),md.reactions.get_by_id("PUNP4"),
         md.reactions.get_by_id("PUNP4_1"),md.reactions.get_by_id("PUNP5"),md.reactions.get_by_id("PUNP5_1"),
         md.reactions.get_by_id("PUNP6"),md.reactions.get_by_id("PUNP6_1"),md.reactions.get_by_id("PUNP7"),
         md.reactions.get_by_id("PUNP7_1"),md.reactions.get_by_id("RECOAH6"),md.reactions.get_by_id("SUCOAS"),
         md.reactions.get_by_id("TDACPT"),md.reactions.get_by_id("TRE6PH"),md.reactions.get_by_id("UAG4Ei"),
         md.reactions.get_by_id("UAGDP"),md.reactions.get_by_id("UCPP"),md.reactions.get_by_id("UDPACGLP"),
         md.reactions.get_by_id("UTCY"),md.reactions.get_by_id("XYLI2"),md.reactions.get_by_id("YUMPS")]

flux_variability_analysis(md,rlist,loopless=True)

,minimum,maximum
3HAACOAT140,-5.767047e-01,-5.767047e-01
3HACPH,0.000000e+00,0.000000e+00
3HAD140,0.000000e+00,0.000000e+00
3HAD40,0.000000e+00,0.000000e+00
3HOXTPP,0.000000e+00,0.000000e+00
...,...,...
UCPP,0.000000e+00,0.000000e+00
UDPACGLP,0.000000e+00,-1.868727e-12
UTCY,0.000000e+00,0.000000e+00
XYLI2,1.423611e-12,0.000000e+00


In [16]:
output = input.replace("xml", "")
output

'/scr/k61san/natasha/matomic/trials/CarveMe/Btheta.tcds.top5.gramNegN.cim8.'

In [17]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)

/scr/k61san/natasha/matomic/trials/CarveMe/Btheta.tcds.top5.gramNegN.cim8.manual.xml
